# Colab → VSCode bridge

Run this notebook **on Colab** (T4 GPU runtime). It clones the stocker repo,
installs deps, then starts a Jupyter server tunneled via cloudflared.

Then on your **local machine**, open VSCode → Command Palette →
*Jupyter: Specify Jupyter Server for Connections* → paste the URL printed
by the last cell. Open `training/train_grpo.ipynb` locally; pick the remote
kernel; cells now execute on Colab's GPU while you edit in VSCode.

Caveat: the kernel runs on Colab, so file paths must match what the kernel
sees. The launcher `cd`s into `/content/stocker` and adds it to `sys.path`
before starting the server, so `import app...` works out of the box.
Files you save inside the notebook write to Colab's filesystem — push them
back via `git push` from a Colab cell, or mount Drive (cell 0).

In [ ]:
# OPTIONAL — mount Drive so artifacts under training/runs/ persist across sessions.
# Skip if you'll just download artifacts at the end of the run.
# from google.colab import drive
# drive.mount('/content/drive')
pass

In [ ]:
import os
REPO_URL = 'https://github.com/<your-username>/stocker.git'   # <-- edit me
WORKDIR  = '/content/stocker'
if not os.path.isdir(WORKDIR):
    assert '<your-username>' not in REPO_URL, 'Edit REPO_URL first.'
    !git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git pull --rebase 2>/dev/null || true

In [ ]:
# Same dep set as train_grpo.ipynb's install cell.
!pip install -q -U 'transformers>=4.55' 'trl>=0.11' 'peft>=0.13' 'accelerate>=1.0' 'bitsandbytes>=0.43' 'datasets>=3.0'
!pip install -q --upgrade-strategy=only-if-needed yfinance mplfinance pyarrow 'pydantic>=2' pydantic-settings 'openai>=1' tensorboard 'huggingface_hub>=1.10' jupyter-server

In [ ]:
# Install cloudflared (free trycloudflare.com tunnel, no signup).
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Pre-build the bundled dataset on Colab so VSCode-driven cells can
# `from app.data import loader` immediately. Idempotent.
!python scripts/build_dataset.py
!python scripts/validate_tasks.py

In [ ]:
import os, secrets, subprocess, time, re, pathlib, signal

TOKEN = secrets.token_urlsafe(24)
PORT  = 9988
LOG_J = pathlib.Path('/content/jupyter.log')
LOG_C = pathlib.Path('/content/cloudflared.log')

# Kill any previous instances
for proc in ('jupyter-server', 'cloudflared'):
    !pkill -f {proc} 2>/dev/null
time.sleep(1)

# Start jupyter-server in the background, listening only on localhost.
# We run it from /content/stocker so the kernel's cwd matches the repo.
os.environ['PYTHONPATH'] = '/content/stocker'
j = subprocess.Popen(
    [
        'jupyter', 'server',
        f'--ServerApp.token={TOKEN}',
        '--ServerApp.password=',
        f'--ServerApp.port={PORT}',
        '--ServerApp.ip=127.0.0.1',
        '--ServerApp.allow_origin=*',
        '--ServerApp.disable_check_xsrf=True',
        '--ServerApp.allow_remote_access=True',
        '--no-browser',
        f'--ServerApp.root_dir=/content/stocker',
    ],
    stdout=open(LOG_J, 'w'),
    stderr=subprocess.STDOUT,
    cwd='/content/stocker',
)
time.sleep(3)
assert j.poll() is None, f'jupyter-server died:\n{LOG_J.read_text()[-2000:]}'
print('jupyter-server up on port', PORT)

# Start the cloudflared quick tunnel.
c = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=open(LOG_C, 'w'),
    stderr=subprocess.STDOUT,
)

# Wait for the public URL to appear in the cloudflared log.
url = None
for _ in range(40):
    time.sleep(1)
    txt = LOG_C.read_text() if LOG_C.exists() else ''
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break
assert url, f'cloudflared did not produce a URL:\n{LOG_C.read_text()[-2000:]}'

FULL = f'{url}/?token={TOKEN}'
print('\n' + '=' * 70)
print('Paste this into VSCode -> Jupyter: Specify Jupyter Server for Connections:')
print()
print('   ', FULL)
print()
print('Or use it from anywhere:  curl -s', f'"{FULL}/api/status"')
print('=' * 70)
print('\nKeep this Colab tab open. The tunnel dies when the runtime disconnects.')

## Connect from VSCode

1. **Install the Jupyter extension** in VSCode (Microsoft, ID `ms-toolsai.jupyter`) if you don't have it.
2. Open `training/train_grpo.ipynb` locally.
3. Command Palette → **`Jupyter: Specify Jupyter Server for Connections`** → *Existing* → paste the URL printed above.
4. In the kernel picker (top-right of the notebook), pick the remote runtime — VSCode shows it as `Python 3 (ipykernel)` under the trycloudflare host.
5. Run cells. Imports like `from app.council.llm import TransformersLLMClient` resolve because the kernel cwd is `/content/stocker`.

## Heartbeat

If you idle, Colab eventually evicts the runtime and the URL stops working.
Re-run the **`launch`** cell to get a new URL and re-specify the server in
VSCode. Long training runs should run inside a `nohup`-style guard or use
Colab Pro for the 24-hour runtime.

## Pulling artifacts back

Trained LoRAs and plots land in `/content/stocker/training/runs/<id>/`. To
get them onto your laptop:

```python
# in a Colab cell, after training
from google.colab import files
import shutil, glob
run = sorted(glob.glob('/content/stocker/training/runs/grpo_*'))[-1]
shutil.make_archive('/content/run', 'zip', run)
files.download('/content/run.zip')
```

Or `git add` + `git push` from a Colab cell to your fork.